In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import os
import time
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import xarray as xr
import cdsapi

# =========================
# USER SETTINGS
# =========================

HYDE_ROOT = Path(".")  # contains general_files/ + gbc2025_7apr_base/
SCENARIO = "base"                   # "base" | "upper" | "lower"

YEAR_START = 1950
YEAR_END   = 2025

# Where temporary ERA5 monthly downloads will be stored (then deleted)
TMP_ERA5_DIR = Path("/tmp/era5_hourly_hyde_regions")

# Output
OUT_PARQUET = Path("hyde_region_year_panel_era5_hourly.parquet")
OUT_CSV_GZ  = Path("hyde_region_year_panel_era5_hourly.csv.gz")

# ERA5 request settings
ERA5_DATASET = "reanalysis-era5-single-levels"
ERA5_VARS = [
    "2m_temperature",       # t2m (K)
    "total_precipitation",  # tp (m)
]

# Optional: reduce size by requesting coarser grid.
# (You can set [0.5,0.5] or [1.0,1.0] if you accept coarser spatial detail.)
ERA5_GRID = None  # e.g. [0.5, 0.5]

# Add padding to HYDE region bounding boxes (degrees)
BBOX_PAD_DEG = 0.5

# Weighting for region means: "cropland" or "population" or "area"
CLIMATE_WEIGHT = "cropland"

# Thresholds / crop-relevant climate risk
HOT_HOUR_THRESHOLD_C = 30.0
GDD_BASE_C = 10.0

# Rolling volatility window (years)
ROLLING_WIN = 10

# =========================
# HYDE FILES (expected)
# =========================

HYDE_GENERAL = HYDE_ROOT / "general_files" / "general_files"
HYDE_SC_DIR  = HYDE_ROOT / f"gbc2025_7apr_{SCENARIO}"

# Rasters (ASCII grids aligned)
IM_REG_ASC   = HYDE_GENERAL / "im_reg_cr.asc"   # HYDE macro regions
ISO_ASC      = HYDE_GENERAL / "iso_cr.asc"      # HYDE country codes (optional parallel later)
GAREA_ASC    = HYDE_GENERAL / "garea_cr.asc"    # grid cell area weights
LANDLAKE_ASC = HYDE_GENERAL / "landlake.asc"    # land mask (optional)

# HYDE NetCDFs (variables used in panel)
HYDE_NC = {
    "population":        HYDE_SC_DIR / "NetCDF/population.nc",
    "cropland":          HYDE_SC_DIR / "NetCDF/cropland.nc",
    "pasture":           HYDE_SC_DIR / "NetCDF/pasture.nc",
    "rangeland":         HYDE_SC_DIR / "NetCDF/rangeland.nc",
    "grazing_land":      HYDE_SC_DIR / "NetCDF/grazing_land.nc",
    "total_rainfed":     HYDE_SC_DIR / "NetCDF/total_rainfed.nc",
    "total_irrigated":   HYDE_SC_DIR / "NetCDF/total_irrigated.nc",
    "total_rice":        HYDE_SC_DIR / "NetCDF/total_rice.nc",
    "rural_population":  HYDE_SC_DIR / "NetCDF/rural_population.nc",
    "urban_population":  HYDE_SC_DIR / "NetCDF/urban_population.nc",
    "urban_area":        HYDE_SC_DIR / "NetCDF/urban_area.nc",
}

# Optional: crop shares / productivity (hook)
CROP_SHARES_FILE = None  # e.g. HYDE_ROOT/"sub_national_data"/"his_crop_4apr2025.csv"
# Optional: external yields dataset path (hook)
YIELDS_FILE = None

# =========================
# Helpers: read ESRI ASCII grid
# =========================

def read_esri_ascii_grid(path: Path) -> Tuple[np.ndarray, dict]:
    meta = {}
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header = [f.readline().strip() for _ in range(6)]
        for ln in header:
            parts = ln.split()
            if len(parts) < 2:
                continue
            k, v = parts[0].lower(), parts[1]
            if k in ("ncols", "nrows"):
                meta[k] = int(float(v))
            elif k in ("xllcorner", "yllcorner", "cellsize"):
                meta[k] = float(v)
            elif k == "nodata_value":
                meta["nodata_value"] = float(v)
        meta.setdefault("nodata_value", -9999.0)
        nrows, ncols = meta["nrows"], meta["ncols"]
        data = np.loadtxt(f, dtype=float).reshape((nrows, ncols))
    return data, meta

def ascii_grid_latlon(meta: dict) -> Tuple[np.ndarray, np.ndarray]:
    nrows, ncols, cell = meta["nrows"], meta["ncols"], meta["cellsize"]
    x0, y0 = meta["xllcorner"], meta["yllcorner"]
    lons = x0 + (np.arange(ncols) + 0.5) * cell
    lats_s2n = y0 + (np.arange(nrows) + 0.5) * cell
    lats = lats_s2n[::-1]  # ASCII rows are north->south
    return lats, lons

def hyde_rowcol_from_latlon(lat2d: np.ndarray, lon2d: np.ndarray, meta: dict) -> Tuple[np.ndarray, np.ndarray]:
    nrows, ncols, cell = meta["nrows"], meta["ncols"], meta["cellsize"]
    x0, y0 = meta["xllcorner"], meta["yllcorner"]

    lon_adj = lon2d.copy()
    if x0 < 0 and np.nanmax(lon_adj) > 180:
        lon_adj = ((lon_adj + 180) % 360) - 180
    if x0 >= 0 and np.nanmin(lon_adj) < 0:
        lon_adj = lon_adj % 360

    col = np.floor((lon_adj - x0) / cell).astype(int)
    row_s2n = np.floor((lat2d - y0) / cell).astype(int)
    row = (nrows - 1) - row_s2n

    inb = (row >= 0) & (row < nrows) & (col >= 0) & (col < ncols)
    row = np.where(inb, row, -1)
    col = np.where(inb, col, -1)
    return row, col

def safe_first_data_var(ds: xr.Dataset) -> str:
    v = list(ds.data_vars)
    if not v:
        raise ValueError("Dataset has no data_vars")
    return v[0]

def select_year(da: xr.DataArray, year: int) -> xr.DataArray:
    if "year" in da.dims:
        return da.sel(year=year)
    if "time" in da.dims:
        try:
            return da.sel(time=year)
        except Exception:
            return da.sel(time=year, method="nearest")
    return da

# =========================
# Define HYDE region bounding boxes
# =========================

@dataclass
class RegionBBox:
    region_hyde: str
    north: float
    west: float
    south: float
    east: float

def build_region_bboxes(im_reg: np.ndarray, meta: dict, pad: float) -> Dict[str, RegionBBox]:
    lats, lons = ascii_grid_latlon(meta)
    nrows, ncols = im_reg.shape
    lat2d = np.repeat(lats[:, None], ncols, axis=1)
    lon2d = np.repeat(lons[None, :], nrows, axis=0)

    out: Dict[str, RegionBBox] = {}
    for code in np.unique(im_reg.astype(int)):
        if code <= 0:
            continue
        m = (im_reg.astype(int) == code)
        if not m.any():
            continue
        lat_vals = lat2d[m]
        lon_vals = lon2d[m]
        north = float(np.max(lat_vals) + pad)
        south = float(np.min(lat_vals) - pad)
        west  = float(np.min(lon_vals) - pad)
        east  = float(np.max(lon_vals) + pad)

        # clip to valid geographic ranges
        north = min(90.0, north); south = max(-90.0, south)
        west = max(-180.0, west); east = min(180.0, east)

        out[str(int(code))] = RegionBBox(str(int(code)), north, west, south, east)
    return out

# =========================
# HYDE aggregation to region-year (sums over cells)
# =========================

def aggregate_hyde_region_year(
    im_reg: np.ndarray,
    landmask: Optional[np.ndarray],
    hyde_arrays_by_year: Dict[int, Dict[str, np.ndarray]],
) -> pd.DataFrame:
    # Flatten once for grouping
    reg_flat = im_reg.astype(int).ravel()
    if landmask is not None:
        land_flat = (landmask.ravel() != 0)
    else:
        land_flat = np.ones_like(reg_flat, dtype=bool)

    rows = []
    for year, vars2d in hyde_arrays_by_year.items():
        # build frame with region and values
        df = pd.DataFrame({"region_hyde": reg_flat, "land": land_flat})
        df = df[df["land"] & (df["region_hyde"] > 0)].copy()
        df["year"] = year

        for k, a in vars2d.items():
            df[k] = a.ravel()[df.index]

        g = df.groupby(["region_hyde", "year"], sort=False)
        # Sum most HYDE land-use/pop variables
        out = g[list(vars2d.keys())].sum(min_count=1).reset_index()
        out["region_hyde"] = out["region_hyde"].astype(int).astype(str)
        rows.append(out)

    return pd.concat(rows, ignore_index=True)

# =========================
# ERA5 hourly -> region-year stats (download per region-month, delete file)
# =========================

@dataclass
class Acc:
    # annual accumulators for region-year
    w_sum: float = 0.0
    t_wsum: float = 0.0
    p_wsum: float = 0.0

    # extremes over the year (hourly)
    t_min: float = np.inf
    t_max: float = -np.inf

    # daily aggregates for volatility
    daily_tmean: List[float] = None
    daily_psum: List[float] = None

    # crop-relevant metrics
    hot_hours: float = 0.0
    hours_total: float = 0.0
    gdd_sum: float = 0.0  # computed from daily mean

    def __post_init__(self):
        if self.daily_tmean is None:
            self.daily_tmean = []
        if self.daily_psum is None:
            self.daily_psum = []

def era5_area_weights(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    # Relative area weights for regular lat-lon: cos(lat)
    w_lat = np.cos(np.deg2rad(lat))
    w_lat = np.clip(w_lat, 0, None)
    return w_lat[:, None] * np.ones((lat.size, lon.size))

def get_hyde_weight_grid(pop2d: np.ndarray, crop2d: np.ndarray, area2d: np.ndarray, mode: str) -> np.ndarray:
    if mode == "population":
        w = pop2d
    elif mode == "cropland":
        w = crop2d
    elif mode == "area":
        w = area2d
    else:
        raise ValueError(f"Unknown CLIMATE_WEIGHT={mode}")
    w = np.where(np.isfinite(w) & (w > 0), w, 0.0)
    return w

def region_year_from_era5_hourly(
    region: RegionBBox,
    year: int,
    client: cdsapi.Client,
    im_reg: np.ndarray,
    landmask: Optional[np.ndarray],
    meta: dict,
    hyde_w2d: np.ndarray,
    tmpdir: Path,
) -> Dict[str, float]:
    acc = Acc()

    times = [f"{h:02d}:00" for h in range(24)]
    days  = [f"{d:02d}" for d in range(1, 32)]  # CDS ignores invalid dates

    for month in range(1, 13):
        tmpdir.mkdir(parents=True, exist_ok=True)
        f = tmpdir / f"era5_{region.region_hyde}_{year:04d}{month:02d}.nc"
        if f.exists():
            f.unlink()  # be strict: rebuild if leftover

        req = {
            "product_type": "reanalysis",
            "variable": ERA5_VARS,
            "year": f"{year:04d}",
            "month": f"{month:02d}",
            "day": days,
            "time": times,
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": [region.north, region.west, region.south, region.east],
        }
        if ERA5_GRID is not None:
            req["grid"] = ERA5_GRID

        # Download with retry
        for attempt in range(1, 9):
            try:
                r = client.retrieve(ERA5_DATASET, req)
                try:
                    r.download(str(f))
                except Exception:
                    client.retrieve(ERA5_DATASET, req, str(f))
                if f.exists() and f.stat().st_size > 0:
                    break
                raise RuntimeError("ERA5 file missing/empty")
            except Exception as e:
                if attempt == 8:
                    raise
                wait = min(900, 10 * (2 ** (attempt - 1)))
                print(f"[err] {region.region_hyde} {year}-{month:02d} attempt {attempt}/8: {e} (sleep {wait}s)")
                time.sleep(wait)

        # Process file, then delete
        ds = xr.open_dataset(f)

        # Robust name resolution
        # In ERA5 single levels: t2m and tp are common short names, but some files store them exactly that way.
        tname = "t2m" if "t2m" in ds.data_vars else None
        pname = "tp"  if "tp"  in ds.data_vars else None
        if tname is None or pname is None:
            # fallback: find by attrs/name
            # if you hit this, print(ds.data_vars) once and hardcode.
            raise KeyError(f"Expected t2m/tp in {f.name}. Found: {list(ds.data_vars)}")

        t2m = ds[tname]  # (time, lat, lon)
        tp  = ds[pname]  # (time, lat, lon)

        lat = t2m["latitude"].values if "latitude" in t2m.coords else t2m["lat"].values
        lon = t2m["longitude"].values if "longitude" in t2m.coords else t2m["lon"].values

        # Convert units: K->C, m->mm per hour (tp is accumulated over step; in hourly, treat as hourly increment)
        T = (t2m - 273.15)
        P = (tp * 1000.0)

        # Map ERA5 grid points to HYDE grid (nearest cell)
        lat2d = np.repeat(lat[:, None], len(lon), axis=1)
        lon2d = np.repeat(lon[None, :], len(lat), axis=0)
        rr, cc = hyde_rowcol_from_latlon(lat2d, lon2d, meta)
        valid = (rr >= 0) & (cc >= 0)

        # Keep only points whose mapped HYDE cell is in this region, and land (optional)
        reg_codes = np.zeros_like(rr, dtype=int)
        reg_codes[valid] = im_reg[rr[valid], cc[valid]].astype(int)
        in_region = valid & (reg_codes == int(region.region_hyde))

        if landmask is not None:
            lm = np.zeros_like(in_region, dtype=bool)
            lm[valid] = (landmask[rr[valid], cc[valid]] != 0)
            in_region &= lm

        if not in_region.any():
            ds.close()
            f.unlink(missing_ok=True)
            continue

        # Weights: ERA5 area weights * HYDE weight (pop/cropland/area) of mapped HYDE cell
        w_area = era5_area_weights(np.asarray(lat, float), np.asarray(lon, float))
        mapped_hyde_w = np.zeros_like(w_area)
        mapped_hyde_w[in_region] = hyde_w2d[rr[in_region], cc[in_region]]
        w2 = w_area * mapped_hyde_w

        # Expand weights to time dimension where needed
        # Compute monthly contribution to annual mean T and annual total P
        # For T: mean over hours within month, then weighted over space.
        # For P: sum over hours within month (mm), then weighted over space.
        # Then combine months by weighted sums (accumulators).
        Tm = T.mean("time")  # (lat,lon)
        Pm = P.sum("time")   # (lat,lon)  monthly mm

        Tm_np = np.asarray(Tm.values, float)
        Pm_np = np.asarray(Pm.values, float)

        msk = (w2 > 0) & np.isfinite(Tm_np) & np.isfinite(Pm_np)

        if msk.any():
            wsum = float(w2[msk].sum())
            tbar = float((Tm_np[msk] * w2[msk]).sum() / wsum)
            psum = float((Pm_np[msk] * w2[msk]).sum() / wsum)

            acc.w_sum += wsum
            acc.t_wsum += wsum * tbar
            acc.p_wsum += wsum * psum

        # Extremes: min/max of hourly temp across region (unweighted; you can weight if you really want)
        # Do it cheaply by masking spatially then reducing.
        # Create a spatial mask index once:
        idx = np.where(in_region)
        # Pull subset lazily
        T_sub = T.isel(latitude=xr.DataArray(idx[0], dims="points"),
                       longitude=xr.DataArray(idx[1], dims="points"))
        # T_sub dims: time x points
        tmin = float(T_sub.min().values)
        tmax = float(T_sub.max().values)
        acc.t_min = min(acc.t_min, tmin)
        acc.t_max = max(acc.t_max, tmax)

        # Crop-relevant: hot hour share
        hot = (T_sub > HOT_HOUR_THRESHOLD_C).sum().values
        tot = np.prod(T_sub.shape)
        acc.hot_hours += float(hot)
        acc.hours_total += float(tot)

        # Daily series for volatility + GDD
        # Convert to daily mean temp and daily precip sum over region (weighted by w2)
        # Use w2 normalized on region points.
        # Build a points weight vector aligned with idx
        w_points = w2[in_region]
        w_points = np.where(np.isfinite(w_points) & (w_points > 0), w_points, 0.0)
        wden = w_points.sum()
        if wden > 0:
            w_points = w_points / wden

            # daily mean temp: average over time within each day, then weighted over points
            # daily precip sum: sum over time within each day, weighted over points
            T_daily_points = T_sub.resample(time="1D").mean()  # day x points
            P_daily_points = P.isel(latitude=xr.DataArray(idx[0], dims="points"),
                                    longitude=xr.DataArray(idx[1], dims="points")).resample(time="1D").sum()

            Td = np.asarray(T_daily_points.values, float)  # ndays x npoints
            Pd = np.asarray(P_daily_points.values, float)

            # weighted over points
            td_series = (Td * w_points[None, :]).sum(axis=1)
            pd_series = (Pd * w_points[None, :]).sum(axis=1)

            acc.daily_tmean.extend(td_series.tolist())
            acc.daily_psum.extend(pd_series.tolist())

            # GDD from daily mean
            gdd = np.maximum(0.0, td_series - GDD_BASE_C).sum()
            acc.gdd_sum += float(gdd)

        ds.close()
        f.unlink(missing_ok=True)

    # finalize annual stats
    if acc.w_sum > 0:
        t_mean = acc.t_wsum / acc.w_sum
        p_sum  = acc.p_wsum / acc.w_sum
    else:
        t_mean = np.nan
        p_sum  = np.nan

    daily_t = np.asarray(acc.daily_tmean, float) if acc.daily_tmean else np.array([], float)
    daily_p = np.asarray(acc.daily_psum, float)  if acc.daily_psum else np.array([], float)

    # vol measures (within-year)
    t_sd_daily = float(np.nanstd(daily_t, ddof=1)) if daily_t.size >= 3 else np.nan
    p_sd_daily = float(np.nanstd(daily_p, ddof=1)) if daily_p.size >= 3 else np.nan
    p_cv_daily = float(p_sd_daily / np.nanmean(daily_p)) if daily_p.size >= 3 and np.nanmean(daily_p) > 0 else np.nan

    hot_share = float(acc.hot_hours / acc.hours_total) if acc.hours_total > 0 else np.nan

    return {
        "region_hyde": region.region_hyde,
        "year": year,
        "t2m_mean_c": t_mean,
        "tp_sum_mm": p_sum,
        "t2m_min_c": (acc.t_min if np.isfinite(acc.t_min) else np.nan),
        "t2m_max_c": (acc.t_max if np.isfinite(acc.t_max) else np.nan),
        "t2m_sd_dailymean": t_sd_daily,
        "tp_sd_dailysum": p_sd_daily,
        "tp_cv_dailysum": p_cv_daily,
        "hot_hour_share": hot_share,
        "gdd_base10": acc.gdd_sum,
        "climate_weight": CLIMATE_WEIGHT,
    }

# =========================
# Optional: crop shares / productivity hooks
# =========================

def compute_crop_shares_region_year_stub() -> Optional[pd.DataFrame]:
    """
    Hook: read HYDE crop tables and aggregate to (region_hyde, year).
    This depends on the exact schema of your HYDE crop CSV(s).
    Return DataFrame with columns: region_hyde, year, share_*.
    """
    return None

def compute_crop_productivity_stub() -> Optional[pd.DataFrame]:
    """
    Hook: merge yields or build additional productivity proxies.
    Return DataFrame keyed by region_hyde, year (or iso3, year).
    """
    return None

# =========================
# Main
# =========================

def main():
    # Preconditions
    for p in [IM_REG_ASC, GAREA_ASC] + list(HYDE_NC.values()):
        if not p.exists():
            raise FileNotFoundError(p)

    im_reg, meta = read_esri_ascii_grid(IM_REG_ASC)
    garea, _ = read_esri_ascii_grid(GAREA_ASC)

    landmask = None
    if LANDLAKE_ASC.exists():
        landmask, _ = read_esri_ascii_grid(LANDLAKE_ASC)

    # Define “reasonable areas” = bounding boxes per HYDE region
    bboxes = build_region_bboxes(im_reg, meta, BBOX_PAD_DEG)
    regions = sorted(bboxes.keys(), key=lambda x: int(x))

    # Load HYDE NetCDFs lazily
    hyde_da = {}
    for k, ncpath in HYDE_NC.items():
        ds = xr.open_dataset(ncpath, decode_times=False)
        hyde_da[k] = ds[safe_first_data_var(ds)]

    # Precompute HYDE region-year aggregates for land use + population
    hyde_arrays_by_year: Dict[int, Dict[str, np.ndarray]] = {}
    for year in range(YEAR_START, YEAR_END + 1):
        hyde_arrays_by_year[year] = {}
        for k, da in hyde_da.items():
            sli = select_year(da, year)
            if sli.ndim == 3:
                sli = sli.isel({sli.dims[0]: 0})
            hyde_arrays_by_year[year][k] = np.asarray(sli.values, float)

    hyde_panel = aggregate_hyde_region_year(im_reg, landmask, hyde_arrays_by_year)

    # ERA5 client
    client = cdsapi.Client()
    TMP_ERA5_DIR.mkdir(parents=True, exist_ok=True)

    # Compute climate stats region-year
    climate_rows = []
    for year in range(YEAR_START, YEAR_END + 1):
        pop2d = hyde_arrays_by_year[year]["population"]
        crop2d = hyde_arrays_by_year[year]["cropland"]
        hyde_w2d = get_hyde_weight_grid(pop2d, crop2d, garea, CLIMATE_WEIGHT)

        for rid in regions:
            rb = bboxes[rid]
            print(f"[climate] region={rid} year={year}")
            row = region_year_from_era5_hourly(
                region=rb,
                year=year,
                client=client,
                im_reg=im_reg,
                landmask=landmask,
                meta=meta,
                hyde_w2d=hyde_w2d,
                tmpdir=TMP_ERA5_DIR,
            )
            climate_rows.append(row)

    climate_panel = pd.DataFrame(climate_rows)

    # Merge HYDE + climate
    panel = hyde_panel.merge(climate_panel, on=["region_hyde", "year"], how="left")

    # Add rolling risk measures (year-to-year volatility) on the merged annual series
    panel = panel.sort_values(["region_hyde", "year"]).copy()
    panel["t2m_sd_roll"] = panel.groupby("region_hyde")["t2m_mean_c"].rolling(ROLLING_WIN, min_periods=max(3, ROLLING_WIN//2)).std().reset_index(level=0, drop=True)
    panel["tp_sd_roll"]  = panel.groupby("region_hyde")["tp_sum_mm"].rolling(ROLLING_WIN, min_periods=max(3, ROLLING_WIN//2)).std().reset_index(level=0, drop=True)

    # Optional crop shares / productivity merges
    cs = compute_crop_shares_region_year_stub()
    if cs is not None:
        panel = panel.merge(cs, on=["region_hyde", "year"], how="left")

    prod = compute_crop_productivity_stub()
    if prod is not None:
        panel = panel.merge(prod, on=["region_hyde", "year"], how="left")

    # Write output
    panel.to_parquet(OUT_PARQUET, index=False)
    panel.to_csv(OUT_CSV_GZ, index=False, compression="gzip")
    print(f"[ok] wrote {OUT_PARQUET}")
    print(f"[ok] wrote {OUT_CSV_GZ}")

    # Clean up tmpdir if empty (optional)
    try:
        if TMP_ERA5_DIR.exists() and not any(TMP_ERA5_DIR.iterdir()):
            TMP_ERA5_DIR.rmdir()
    except Exception:
        pass

if __name__ == "__main__":
    main()

with storage

In [2]:
#!/usr/bin/env python3
from __future__ import annotations

import time
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import xarray as xr
import cdsapi

# =========================
# USER SETTINGS
# =========================

HYDE_ROOT = Path(".")
SCENARIO = "base"
YEAR_START = 1950
YEAR_END   = 2025

# Persistent ERA5 store (NOT deleted)
ERA5_STORE = Path("ERA5")

# Final outputs
OUT_PARQUET = Path("hyde_region_year_panel_era5_hourly.parquet")
OUT_CSV_GZ  = Path("hyde_region_year_panel_era5_hourly.csv.gz")

# ERA5 request
ERA5_DATASET = "reanalysis-era5-single-levels"
ERA5_VARS = ["2m_temperature", "total_precipitation"]
ERA5_GRID = None           # e.g. [0.5, 0.5]
BBOX_PAD_DEG = 0.5

# region averaging weights for climate: "cropland" | "population" | "area"
CLIMATE_WEIGHT = "cropland"

# crop-relevant metrics
HOT_HOUR_THRESHOLD_C = 30.0
GDD_BASE_C = 10.0

# rolling year-to-year risk window
ROLLING_WIN = 10

# =========================
# HYDE inputs
# =========================

HYDE_GENERAL = HYDE_ROOT / "general_files" / "general_files"
HYDE_SC_DIR  = HYDE_ROOT / f"gbc2025_7apr_{SCENARIO}"

IM_REG_ASC   = HYDE_GENERAL / "im_reg_cr.asc"
GAREA_ASC    = HYDE_GENERAL / "garea_cr.asc"
LANDLAKE_ASC = HYDE_GENERAL / "landlake.asc"

HYDE_NC = {
    "population":        HYDE_SC_DIR / "NetCDF/population.nc",
    "cropland":          HYDE_SC_DIR / "NetCDF/cropland.nc",
    "pasture":           HYDE_SC_DIR / "NetCDF/pasture.nc",
    "rangeland":         HYDE_SC_DIR / "NetCDF/rangeland.nc",
    "grazing_land":      HYDE_SC_DIR / "NetCDF/grazing_land.nc",
    "total_rainfed":     HYDE_SC_DIR / "NetCDF/total_rainfed.nc",
    "total_irrigated":   HYDE_SC_DIR / "NetCDF/total_irrigated.nc",
    "total_rice":        HYDE_SC_DIR / "NetCDF/total_rice.nc",
    "rural_population":  HYDE_SC_DIR / "NetCDF/rural_population.nc",
    "urban_population":  HYDE_SC_DIR / "NetCDF/urban_population.nc",
    "urban_area":        HYDE_SC_DIR / "NetCDF/urban_area.nc",
}

# =========================
# ASCII + grid helpers
# =========================

def read_esri_ascii_grid(path: Path) -> Tuple[np.ndarray, dict]:
    meta = {}
    with path.open("r", encoding="utf-8", errors="ignore") as f:
        header = [f.readline().strip() for _ in range(6)]
        for ln in header:
            parts = ln.split()
            if len(parts) < 2:
                continue
            k, v = parts[0].lower(), parts[1]
            if k in ("ncols", "nrows"):
                meta[k] = int(float(v))
            elif k in ("xllcorner", "yllcorner", "cellsize"):
                meta[k] = float(v)
            elif k == "nodata_value":
                meta["nodata_value"] = float(v)
        meta.setdefault("nodata_value", -9999.0)
        nrows, ncols = meta["nrows"], meta["ncols"]
        data = np.loadtxt(f, dtype=float).reshape((nrows, ncols))
    return data, meta

def ascii_grid_latlon(meta: dict) -> Tuple[np.ndarray, np.ndarray]:
    nrows, ncols, cell = meta["nrows"], meta["ncols"], meta["cellsize"]
    x0, y0 = meta["xllcorner"], meta["yllcorner"]
    lons = x0 + (np.arange(ncols) + 0.5) * cell
    lats_s2n = y0 + (np.arange(nrows) + 0.5) * cell
    lats = lats_s2n[::-1]
    return lats, lons

def hyde_rowcol_from_latlon(lat2d: np.ndarray, lon2d: np.ndarray, meta: dict) -> Tuple[np.ndarray, np.ndarray]:
    nrows, ncols, cell = meta["nrows"], meta["ncols"], meta["cellsize"]
    x0, y0 = meta["xllcorner"], meta["yllcorner"]

    lon_adj = lon2d.copy()
    if x0 < 0 and np.nanmax(lon_adj) > 180:
        lon_adj = ((lon_adj + 180) % 360) - 180
    if x0 >= 0 and np.nanmin(lon_adj) < 0:
        lon_adj = lon_adj % 360

    col = np.floor((lon_adj - x0) / cell).astype(int)
    row_s2n = np.floor((lat2d - y0) / cell).astype(int)
    row = (nrows - 1) - row_s2n

    inb = (row >= 0) & (row < nrows) & (col >= 0) & (col < ncols)
    row = np.where(inb, row, -1)
    col = np.where(inb, col, -1)
    return row, col

def safe_first_data_var(ds: xr.Dataset) -> str:
    v = list(ds.data_vars)
    if not v:
        raise ValueError("Dataset has no data_vars")
    return v[0]

def select_year(da: xr.DataArray, year: int) -> xr.DataArray:
    if "year" in da.dims:
        return da.sel(year=year)
    if "time" in da.dims:
        try:
            return da.sel(time=year)
        except Exception:
            return da.sel(time=year, method="nearest")
    return da

# =========================
# HYDE regions -> bounding boxes
# =========================

@dataclass
class RegionBBox:
    region_hyde: str
    north: float
    west: float
    south: float
    east: float

def build_region_bboxes(im_reg: np.ndarray, meta: dict, pad: float) -> Dict[str, RegionBBox]:
    lats, lons = ascii_grid_latlon(meta)
    nrows, ncols = im_reg.shape
    lat2d = np.repeat(lats[:, None], ncols, axis=1)
    lon2d = np.repeat(lons[None, :], nrows, axis=0)

    out: Dict[str, RegionBBox] = {}
    for code in np.unique(im_reg.astype(int)):
        if code <= 0:
            continue
        m = (im_reg.astype(int) == code)
        if not m.any():
            continue
        lat_vals = lat2d[m]
        lon_vals = lon2d[m]
        north = float(np.max(lat_vals) + pad)
        south = float(np.min(lat_vals) - pad)
        west  = float(np.min(lon_vals) - pad)
        east  = float(np.max(lon_vals) + pad)

        north = min(90.0, north); south = max(-90.0, south)
        west = max(-180.0, west); east = min(180.0, east)

        out[str(int(code))] = RegionBBox(str(int(code)), north, west, south, east)
    return out

# =========================
# ERA5 download (store files)
# =========================

def backoff(attempt: int) -> int:
    return min(900, 10 * (2 ** (attempt - 1)))

def era5_target_path(region_id: str, year: int, month: int) -> Path:
    return ERA5_STORE / f"region_hyde={region_id}" / f"year={year:04d}" / f"era5_sl_hourly_{year:04d}{month:02d}.nc"

def download_era5_region_month(client: cdsapi.Client, bbox: RegionBBox, year: int, month: int) -> Path:
    target = era5_target_path(bbox.region_hyde, year, month)
    target.parent.mkdir(parents=True, exist_ok=True)

    if target.exists() and target.stat().st_size > 0:
        print(f"[skip] {target}")
        return target

    times = [f"{h:02d}:00" for h in range(24)]
    days  = [f"{d:02d}" for d in range(1, 32)]

    req = {
        "product_type": "reanalysis",
        "variable": ERA5_VARS,
        "year": f"{year:04d}",
        "month": f"{month:02d}",
        "day": days,
        "time": times,
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [bbox.north, bbox.west, bbox.south, bbox.east],
    }
    if ERA5_GRID is not None:
        req["grid"] = ERA5_GRID

    for attempt in range(1, 9):
        try:
            print(f"[get] {bbox.region_hyde} {year:04d}-{month:02d} -> {target.name}")
            r = client.retrieve(ERA5_DATASET, req)
            try:
                r.download(str(target))
            except Exception:
                client.retrieve(ERA5_DATASET, req, str(target))
            if target.exists() and target.stat().st_size > 0:
                return target
            raise RuntimeError("Downloaded file missing/empty")
        except Exception as e:
            if attempt == 8:
                raise
            wait = backoff(attempt)
            print(f"[err] attempt {attempt}/8: {e} (sleep {wait}s)")
            time.sleep(wait)

    return target  # unreachable

def phase_download_all():
    im_reg, meta = read_esri_ascii_grid(IM_REG_ASC)
    bboxes = build_region_bboxes(im_reg, meta, BBOX_PAD_DEG)
    regions = sorted(bboxes.keys(), key=lambda x: int(x))

    client = cdsapi.Client()

    for year in range(YEAR_START, YEAR_END + 1):
        for rid in regions:
            bbox = bboxes[rid]
            for month in range(1, 13):
                download_era5_region_month(client, bbox, year, month)

# =========================
# Compute region-year climate stats from stored ERA5 files
# =========================

def era5_area_weights(lat: np.ndarray, lon: np.ndarray) -> np.ndarray:
    w_lat = np.cos(np.deg2rad(lat))
    w_lat = np.clip(w_lat, 0, None)
    return w_lat[:, None] * np.ones((lat.size, lon.size))

def get_hyde_weight_grid(pop2d: np.ndarray, crop2d: np.ndarray, area2d: np.ndarray, mode: str) -> np.ndarray:
    if mode == "population":
        w = pop2d
    elif mode == "cropland":
        w = crop2d
    elif mode == "area":
        w = area2d
    else:
        raise ValueError(mode)
    return np.where(np.isfinite(w) & (w > 0), w, 0.0)

@dataclass
class Acc:
    w_sum: float = 0.0
    t_wsum: float = 0.0
    p_wsum: float = 0.0
    t_min: float = np.inf
    t_max: float = -np.inf
    hot_hours: float = 0.0
    hours_total: float = 0.0
    daily_tmean: List[float] = None
    daily_psum: List[float] = None
    gdd_sum: float = 0.0
    def __post_init__(self):
        self.daily_tmean = [] if self.daily_tmean is None else self.daily_tmean
        self.daily_psum  = [] if self.daily_psum  is None else self.daily_psum

def compute_region_year_from_store(
    region_id: str,
    year: int,
    bbox: RegionBBox,
    im_reg: np.ndarray,
    landmask: Optional[np.ndarray],
    meta: dict,
    hyde_w2d: np.ndarray,
) -> dict:
    acc = Acc()

    for month in range(1, 13):
        f = era5_target_path(region_id, year, month)
        if not f.exists():
            # allow incomplete tails; but you probably want to enforce completeness
            continue

        ds = xr.open_dataset(f)

        if "t2m" not in ds.data_vars or "tp" not in ds.data_vars:
            raise KeyError(f"{f} missing t2m/tp; found {list(ds.data_vars)}")

        t2m = ds["t2m"]
        tp  = ds["tp"]

        lat = t2m["latitude"].values if "latitude" in t2m.coords else t2m["lat"].values
        lon = t2m["longitude"].values if "longitude" in t2m.coords else t2m["lon"].values

        T = (t2m - 273.15)
        P = (tp * 1000.0)

        lat2d = np.repeat(lat[:, None], len(lon), axis=1)
        lon2d = np.repeat(lon[None, :], len(lat), axis=0)
        rr, cc = hyde_rowcol_from_latlon(lat2d, lon2d, meta)
        valid = (rr >= 0) & (cc >= 0)

        reg_codes = np.zeros_like(rr, dtype=int)
        reg_codes[valid] = im_reg[rr[valid], cc[valid]].astype(int)
        in_region = valid & (reg_codes == int(region_id))

        if landmask is not None:
            lm = np.zeros_like(in_region, dtype=bool)
            lm[valid] = (landmask[rr[valid], cc[valid]] != 0)
            in_region &= lm

        if not in_region.any():
            ds.close()
            continue

        w_area = era5_area_weights(np.asarray(lat, float), np.asarray(lon, float))
        mapped_hyde_w = np.zeros_like(w_area)
        mapped_hyde_w[in_region] = hyde_w2d[rr[in_region], cc[in_region]]
        w2 = w_area * mapped_hyde_w

        # monthly mean temp, monthly precip sum
        Tm = np.asarray(T.mean("time").values, float)
        Pm = np.asarray(P.sum("time").values, float)
        msk = (w2 > 0) & np.isfinite(Tm) & np.isfinite(Pm)
        if msk.any():
            wsum = float(w2[msk].sum())
            tbar = float((Tm[msk] * w2[msk]).sum() / wsum)
            psum = float((Pm[msk] * w2[msk]).sum() / wsum)
            acc.w_sum += wsum
            acc.t_wsum += wsum * tbar
            acc.p_wsum += wsum * psum

        # extremes + hot share via point-subset
        idx = np.where(in_region)
        T_sub = T.isel(
            latitude=xr.DataArray(idx[0], dims="points"),
            longitude=xr.DataArray(idx[1], dims="points"),
        )
        acc.t_min = min(acc.t_min, float(T_sub.min().values))
        acc.t_max = max(acc.t_max, float(T_sub.max().values))

        acc.hot_hours += float((T_sub > HOT_HOUR_THRESHOLD_C).sum().values)
        acc.hours_total += float(np.prod(T_sub.shape))

        # daily series (weighted)
        w_points = w2[in_region]
        w_points = np.where(np.isfinite(w_points) & (w_points > 0), w_points, 0.0)
        wden = w_points.sum()
        if wden > 0:
            w_points = w_points / wden
            T_daily_points = T_sub.resample(time="1D").mean()
            P_sub = P.isel(
                latitude=xr.DataArray(idx[0], dims="points"),
                longitude=xr.DataArray(idx[1], dims="points"),
            )
            P_daily_points = P_sub.resample(time="1D").sum()

            Td = np.asarray(T_daily_points.values, float)
            Pd = np.asarray(P_daily_points.values, float)

            td_series = (Td * w_points[None, :]).sum(axis=1)
            pd_series = (Pd * w_points[None, :]).sum(axis=1)

            acc.daily_tmean.extend(td_series.tolist())
            acc.daily_psum.extend(pd_series.tolist())
            acc.gdd_sum += float(np.maximum(0.0, td_series - GDD_BASE_C).sum())

        ds.close()

    if acc.w_sum > 0:
        t_mean = acc.t_wsum / acc.w_sum
        p_sum  = acc.p_wsum / acc.w_sum
    else:
        t_mean = np.nan
        p_sum  = np.nan

    daily_t = np.asarray(acc.daily_tmean, float) if acc.daily_tmean else np.array([], float)
    daily_p = np.asarray(acc.daily_psum, float)  if acc.daily_psum else np.array([], float)

    t_sd_daily = float(np.nanstd(daily_t, ddof=1)) if daily_t.size >= 3 else np.nan
    p_sd_daily = float(np.nanstd(daily_p, ddof=1)) if daily_p.size >= 3 else np.nan
    p_cv_daily = float(p_sd_daily / np.nanmean(daily_p)) if daily_p.size >= 3 and np.nanmean(daily_p) > 0 else np.nan

    hot_share = float(acc.hot_hours / acc.hours_total) if acc.hours_total > 0 else np.nan

    return {
        "region_hyde": region_id,
        "year": year,
        "t2m_mean_c": t_mean,
        "tp_sum_mm": p_sum,
        "t2m_min_c": (acc.t_min if np.isfinite(acc.t_min) else np.nan),
        "t2m_max_c": (acc.t_max if np.isfinite(acc.t_max) else np.nan),
        "t2m_sd_dailymean": t_sd_daily,
        "tp_sd_dailysum": p_sd_daily,
        "tp_cv_dailysum": p_cv_daily,
        "hot_hour_share": hot_share,
        "gdd_base10": acc.gdd_sum,
        "climate_weight": CLIMATE_WEIGHT,
    }

# =========================
# HYDE region-year panel + merge
# =========================

def aggregate_hyde_region_year(im_reg: np.ndarray, landmask: Optional[np.ndarray], by_year_2d: Dict[int, Dict[str, np.ndarray]]) -> pd.DataFrame:
    reg_flat = im_reg.astype(int).ravel()
    land_flat = (landmask.ravel() != 0) if landmask is not None else np.ones_like(reg_flat, bool)

    rows = []
    for year, vars2d in by_year_2d.items():
        df = pd.DataFrame({"region_hyde": reg_flat, "land": land_flat})
        df = df[df["land"] & (df["region_hyde"] > 0)].copy()
        df["year"] = year
        for k, a in vars2d.items():
            df[k] = a.ravel()[df.index]
        g = df.groupby(["region_hyde","year"], sort=False)
        out = g[list(vars2d.keys())].sum(min_count=1).reset_index()
        out["region_hyde"] = out["region_hyde"].astype(int).astype(str)
        rows.append(out)
    return pd.concat(rows, ignore_index=True)

def main_compute_only():
    # load HYDE rasters
    im_reg, meta = read_esri_ascii_grid(IM_REG_ASC)
    garea, _ = read_esri_ascii_grid(GAREA_ASC)
    landmask = None
    if LANDLAKE_ASC.exists():
        landmask, _ = read_esri_ascii_grid(LANDLAKE_ASC)

    bboxes = build_region_bboxes(im_reg, meta, BBOX_PAD_DEG)
    regions = sorted(bboxes.keys(), key=lambda x: int(x))

    # load HYDE NetCDFs lazily
    hyde_da = {}
    for k, ncpath in HYDE_NC.items():
        ds = xr.open_dataset(ncpath, decode_times=False)
        hyde_da[k] = ds[safe_first_data_var(ds)]

    # slice HYDE to 2D arrays per year (for land-use panel + weights)
    by_year_2d: Dict[int, Dict[str, np.ndarray]] = {}
    for year in range(YEAR_START, YEAR_END + 1):
        by_year_2d[year] = {}
        for k, da in hyde_da.items():
            sli = select_year(da, year)
            if sli.ndim == 3:
                sli = sli.isel({sli.dims[0]: 0})
            by_year_2d[year][k] = np.asarray(sli.values, float)

    hyde_panel = aggregate_hyde_region_year(im_reg, landmask, by_year_2d)

    climate_rows = []
    for year in range(YEAR_START, YEAR_END + 1):
        pop2d = by_year_2d[year]["population"]
        crop2d = by_year_2d[year]["cropland"]
        hyde_w2d = get_hyde_weight_grid(pop2d, crop2d, garea, CLIMATE_WEIGHT)

        for rid in regions:
            row = compute_region_year_from_store(
                region_id=rid,
                year=year,
                bbox=bboxes[rid],
                im_reg=im_reg,
                landmask=landmask,
                meta=meta,
                hyde_w2d=hyde_w2d,
            )
            climate_rows.append(row)

    climate_panel = pd.DataFrame(climate_rows)
    panel = hyde_panel.merge(climate_panel, on=["region_hyde","year"], how="left").sort_values(["region_hyde","year"]).copy()

    # rolling risk on annual series
    panel["t2m_sd_roll"] = panel.groupby("region_hyde")["t2m_mean_c"].rolling(ROLLING_WIN, min_periods=max(3, ROLLING_WIN//2)).std().reset_index(level=0, drop=True)
    panel["tp_sd_roll"]  = panel.groupby("region_hyde")["tp_sum_mm"].rolling(ROLLING_WIN, min_periods=max(3, ROLLING_WIN//2)).std().reset_index(level=0, drop=True)

    panel.to_parquet(OUT_PARQUET, index=False)
    panel.to_csv(OUT_CSV_GZ, index=False, compression="gzip")
    print(f"[ok] wrote {OUT_PARQUET}")
    print(f"[ok] wrote {OUT_CSV_GZ}")

# =========================
# Entry point
# =========================

if __name__ == "__main__":
    MODE = "both"  # change to "download" or "both"

    if MODE in ("download", "both"):
        phase_download_all()
    if MODE in ("compute", "both"):
        main_compute_only()

[get] 1 1950-01 -> era5_sl_hourly_195001.nc


2026-02-21 16:56:19,617 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-02-21 16:56:19,618 INFO Request ID is 1bddfa29-3beb-4a67-adfb-093643305f5a
2026-02-21 16:56:19,812 INFO status has been updated to accepted
2026-02-21 16:56:28,757 INFO status has been updated to running
2026-02-21 16:59:13,528 INFO status has been updated to successful


d0006cece474bedd14673f1aacd532f9.zip:   0%|          | 0.00/116M [00:00<?, ?B/s]

[get] 1 1950-02 -> era5_sl_hourly_195002.nc


2026-02-21 16:59:23,144 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-02-21 16:59:23,145 INFO Request ID is 7ef3dbd9-9f93-402c-9e25-bd8e02014411
2026-02-21 16:59:23,347 INFO status has been updated to accepted
2026-02-21 16:59:37,508 INFO status has been updated to running
2026-02-21 17:02:17,094 INFO status has been updated to successful


...

2026-02-23 07:14:44,816 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-02-23 07:14:44,817 INFO Request ID is bdfc3121-4c36-4bc2-aa02-6d91364f8e9e
2026-02-23 07:14:45,006 INFO status has been updated to accepted
2026-02-23 07:15:06,908 INFO status has been updated to running
2026-02-23 07:17:38,815 INFO status has been updated to successful


a4f87ce6bdd27c104402597a8d136119.zip:   0%|          | 0.00/179M [00:00<?, ?B/s]

[get] 2 1952-04 -> era5_sl_hourly_195204.nc


2026-02-23 07:17:53,027 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2026-02-23 07:17:53,029 INFO Request ID is 54861626-88f6-40d7-b2cc-b65599c8458c
2026-02-23 07:17:53,216 INFO status has been updated to accepted
2026-02-23 07:18:07,711 INFO status has been updated to running


KeyboardInterrupt: 